# Modelos predictivos, logística bayesiana e interpretación

**Dueño:** Juanes · Objetivo específico 1 y objetivo general

Este notebook solo **lee y muestra** resultados. Los cálculos viven en los scripts (`scripts/entrenar.py`,
`scripts/comparar.py`, `scripts/bayes.py`, `scripts/interpretar.py`) y en `src/churn/`. Para regenerar
todo, ver la tabla "Reproducir todo desde cero" del `README.md`.

In [1]:
import sys; sys.path.insert(0, '../src')
import json
import pandas as pd
from churn import config

pd.set_option('display.width', 140)
T = config.TABLAS
print('Variables del modelo:', config.VARIABLES_MODELO)

Variables del modelo: ['fallas_llamada', 'quejas', 'segundos_uso', 'frecuencia_sms', 'numeros_distintos', 'meses_suscripcion', 'nivel_cobro', 'plan_tarifa', 'estado', 'edad']


## 1. Comparación de modelos en validación cruzada (solo entrenamiento)

12 configuraciones: 4 técnicas × 3 conjuntos de variables, 5 pliegues estratificados.

In [2]:
cv = pd.read_csv(T / 'comparacion_modelos.csv')
print(cv[['modelo', 'conjunto', 'auc_pr_media', 'auc_pr_desv', 'auc_roc_media']].round(3).to_string(index=False))

   modelo          conjunto  auc_pr_media  auc_pr_desv  auc_roc_media
logistica          completo         0.741        0.046          0.929
    arbol          completo         0.691        0.039          0.904
   bosque          completo         0.894        0.048          0.977
 boosting          completo         0.890        0.030          0.974
logistica sin_estado_quejas         0.518        0.033          0.865
    arbol sin_estado_quejas         0.465        0.059          0.835
   bosque sin_estado_quejas         0.805        0.050          0.959
 boosting sin_estado_quejas         0.790        0.051          0.951
logistica          solo_uso         0.505        0.039          0.849
    arbol          solo_uso         0.447        0.059          0.831
   bosque          solo_uso         0.660        0.069          0.911
 boosting          solo_uso         0.590        0.075          0.887


![Comparación en validación cruzada](../reports/figures/comparacion_cv_auc_pr.png)

## 2. Evaluación única en los 713 clientes de prueba

In [3]:
pr = pd.read_csv(T / 'metricas_prueba_todos.csv')
cols = ['modelo', 'conjunto', 'auc_pr', 'auc_roc', 'captura_20pct', 'captura_con_n_R2', 'falsos_positivos', 'falsos_negativos']
print(pr[pr['conjunto'] == 'completo'][cols].round(3).to_string(index=False))
print()
print(pd.read_csv(T / 'reglas_referencia.csv').round(3).to_string(index=False))

             modelo conjunto  auc_pr  auc_roc  captura_20pct  captura_con_n_R2  falsos_positivos  falsos_negativos
          logistica completo   0.767    0.930          0.741             0.857                11                58
              arbol completo   0.725    0.900          0.768             0.875                14                46
             bosque completo   0.907    0.975          0.911             0.955                 8                26
           boosting completo   0.895    0.974          0.902             0.964                12                28
logistica_bayesiana completo   0.765    0.929          0.732             0.857                11                58

                 regla  contactados   pct  captura  precision
            R1: quejas           64 0.090    0.482      0.844
R2: quejas o no activo          190 0.266    0.866      0.511


![Curvas ROC y precisión-sensibilidad](../reports/figures/curvas_roc_pr.png)

## 3. Criterios del Avance 2 y elección del modelo

In [4]:
crit = json.load(open(T / 'criterios_avance2.json'))
print(json.dumps(crit, indent=2, ensure_ascii=False))

{
  "bosque": {
    "C1_auc_pr_prueba": 0.9067192977333818,
    "C1_cumple": true,
    "C2_ventaja_vs_R2_pp": 8.92857142857143,
    "C2_ventaja_vs_R1_pp": 7.142857142857145,
    "C2_cumple": true,
    "C3_auc_pr_sin_estado_quejas": 0.8038049259269995,
    "C3_cumple": true,
    "C4_desv_pliegues": 0.048337666521765944,
    "C4_dif_cv_prueba": 0.012803270658622745,
    "C4_cumple": true
  },
  "logistica": {
    "C1_auc_pr_prueba": 0.7672848424555221,
    "C1_cumple": true,
    "C2_ventaja_vs_R2_pp": -0.8928571428571508,
    "C2_ventaja_vs_R1_pp": 0.0,
    "C2_cumple": false,
    "C3_auc_pr_sin_estado_quejas": 0.4608397397933404,
    "C3_cumple": true,
    "C4_desv_pliegues": 0.04568927100673431,
    "C4_dif_cv_prueba": 0.026403308747229515,
    "C4_cumple": true
  },
  "logistica_bayesiana": {
    "C1_auc_pr_prueba": 0.7648449680121718,
    "C1_cumple": true,
    "C2_ventaja_vs_R2_pp": -0.8928571428571508,
    "C2_ventaja_vs_R1_pp": 0.0,
    "C2_cumple": false
  },
  "eleccion": {
    

El bosque aleatorio supera a la logística por 0.153 de AUC-PR en validación cruzada (más que el margen
de 0.02 fijado de antemano) y es el único que cumple C1–C4. Es el modelo final (`predicciones_final.csv`).

## 4. ¿En qué medida? Sensibilidad sin queja ni estado

In [5]:
print(pd.read_csv(T / 'sensibilidad_variables.csv').round(3).to_string(index=False))

   modelo          conjunto  auc_pr_media  auc_pr_desv  auc_roc  auc_pr  captura_20pct
logistica          completo         0.741        0.046    0.930   0.767          0.741
   bosque          completo         0.894        0.048    0.975   0.907          0.911
logistica sin_estado_quejas         0.518        0.033    0.841   0.461          0.527
   bosque sin_estado_quejas         0.805        0.050    0.960   0.804          0.884
logistica          solo_uso         0.505        0.039    0.826   0.456          0.509
   bosque          solo_uso         0.660        0.069    0.911   0.673          0.670


## 5. Qué usa el modelo final: importancia por permutación

In [6]:
print(pd.read_csv(T / 'importancia_permutacion.csv').round(3).to_string(index=False))

         variable               nombre  caida_auc_pr_media  caida_auc_pr_desv
           quejas                Queja               0.328              0.040
           estado   Estado (no activo)               0.206              0.018
     segundos_uso      Segundos de uso               0.095              0.013
meses_suscripcion Meses de suscripción               0.069              0.010
numeros_distintos    Números distintos               0.048              0.013
             edad                 Edad               0.046              0.011
   fallas_llamada    Fallas de llamada               0.030              0.007
   frecuencia_sms                  SMS               0.022              0.009
      nivel_cobro       Nivel de cobro               0.003              0.003
      plan_tarifa      Plan (contrato)               0.000              0.001


![Importancia por permutación](../reports/figures/importancia.png)

![Árbol de decisión de profundidad 4](../reports/figures/arbol.png)

## 6. Logística bayesiana: razones de momios con intervalo de credibilidad del 95 %

In [7]:
b = pd.read_csv(T / 'bayes_resumen.csv')
b = b[b['parametro'] != 'intercepto']
print(b[['parametro', 'lectura', 'or_unidad', 'or_unidad_q2.5', 'or_unidad_q97.5', 'rhat']].round(3).to_string(index=False))
print()
print(pd.read_csv(T / 'bayes_diagnosticos.csv').round(3).to_string(index=False))

        parametro                                      lectura  or_unidad  or_unidad_q2.5  or_unidad_q97.5  rhat
   fallas_llamada          por cada falla de llamada adicional      1.086           1.052            1.122 1.002
           quejas                 con queja frente a sin queja     35.308          20.425           61.977 1.003
     segundos_uso por cada 1000 segundos (≈ 17 min) más de uso      0.837           0.741            0.940 1.001
   frecuencia_sms                          por cada 10 SMS más      0.874           0.812            0.928 1.003
numeros_distintos            por cada 10 números distintos más      0.666           0.530            0.815 1.001
meses_suscripcion          por cada 6 meses más de suscripción      0.836           0.744            0.939 1.002
      nivel_cobro                  por cada nivel de cobro más      0.723           0.547            0.931 1.001
      plan_tarifa               contrato frente a pago por uso      0.826           0.169       

![Densidades posteriores de las razones de momios](../reports/figures/bayes_densidades_or.png)

## 7. Bayesiana frente a clásica

In [8]:
print(pd.read_csv(T / 'bayes_vs_clasica_metricas.csv')[['modelo', 'auc_roc', 'auc_pr', 'captura_20pct', 'brier']].round(3).to_string(index=False))

             modelo  auc_roc  auc_pr  captura_20pct  brier
logistica_bayesiana    0.929   0.765          0.732  0.068
  logistica_clasica    0.930   0.766          0.723  0.067


![Razones de momios: bayesiana frente a clásica](../reports/figures/bayes_vs_clasica_or.png)

Resultados completos e interpretación: `docs/bayes_resultados.md`.